### Ejercicio 1 — Cargar y explorar 

In [0]:
from pyspark.sql.functions import count, avg, sum,max

df_ej01 = spark.read.table("samples.nyctaxi.trips")
# Muestra las primeras 10 filas
display(df_ej01.limit(10))
# Muestra los tipos de datos de las columnas
df_ej01.printSchema()
# Muestra la cantidad de filas
df_ej01.count()

### Ejercicio 2 — Seleccionar columnas 

¿Por qué crees que en Big Data es útil seleccionar solo las 
columnas que necesitamos en vez de trabajar siempre con todas?

R/ Para reducir el costo d procesamiento, ya que reducir los datos a solo no necesario hace más eficiente  


In [0]:
# Crea un dataframe a partir de las columnas de otro
df_viajes = df_ej01.select( "trip_distance", "fare_amount", "pickup_zip", "dropoff_zip" )

display(df_viajes.limit(5))

#### Ejercicio 3 — Filtrar datos

De 
los filtros que aplicaste con .count(), ¿cuál devolvió menos resultados? ¿Por qué crees 
lógica o matemáticamente que es así?

R/ Por inferencia mía diría que al pedir más condiciones, las coincidencias son menos y hace que el count sea menor

In [0]:
df_viajes_largos = df_viajes.filter(df_viajes.trip_distance > 5)

display(df_viajes_largos.limit(5))

df_viajes_baratos = df_viajes.filter(df_viajes.fare_amount <= 10).count()

display(df_viajes_baratos)



In [0]:
df_viajes_caros_largos = df_viajes.filter((df_viajes.trip_distance > 2) & (df_viajes.fare_amount > 15)).count()

display(df_viajes_caros_largos)



#### Ejercicio 4 — Crear columnas calculadas
Observando los resultados, ¿crees que la tarifa_total refleja 
el costo real final que pagó el pasajero? ¿Qué otros cargos comunes en los taxis (que no 
tenemos en estas columnas) podrían faltar? 


In [0]:
# a) Agrega columna impuesto_estimado
df_impuestos = df_viajes.withColumn("impuesto_estimado",  df_viajes["fare_amount"]* 0.09)
display(df_impuestos.limit(5))

# b) Agrega columna tarifa_total
df_costos = df_impuestos.withColumn("tarifa_total", df_viajes["fare_amount"] + df_impuestos["impuesto_estimado"])

# c) Muestra 10 filas seleccionando solo las columnas indicadas
display(df_costos.select("trip_distance", "fare_amount", "impuesto_estimado", "tarifa_total").limit(10))




#### Ejercicio 5 — Agrupar y agregar
Según tu tabla resultante, ¿cuál es el código postal (pickup_zip) de donde salen 
más viajes? ¿Tiene ese código postal la tarifa promedio más alta?
R/ No, El codigo posta de donde hay mas viajes no tiene la tarifa promedio mas alta 


In [0]:
from pyspark.sql.functions import round, max

df_grouped = df_ej01.groupBy("pickup_zip").agg(
    count("*").alias("num_viajes"),
    round(avg("trip_distance"), 2).alias("avg_trip_distance"),
    round(avg("fare_amount"), 2).alias("avg_fare_amount")
)

display(df_grouped.select("pickup_zip", "num_viajes").orderBy("num_viajes", ascending=False).limit(1))

In [0]:
# Encuentra el código postal con la tarifa promedio más alta
display(df_grouped.select("pickup_zip", "avg_fare_amount").orderBy("avg_fare_amount", ascending=False).limit(1))

#### Ejercicio 6 — Consulta SQL equivalente

In [0]:
%sql
SELECT
  pickup_zip,
  COUNT(*) AS num_trips,
  round(AVG(trip_distance),2) AS avg_distance,
  round(AVG(fare_amount),2) AS avg_fare
FROM samples.nyctaxi.trips
GROUP BY pickup_zip
ORDER BY num_trips DESC
LIMIT 5

#### Ejercicio 7 (BONUS)

In [0]:
# Calcula el número de viajes por ruta y los ordena de mayor a menor y muestra los 5 más frecuentes
df_ej07 = df_ej01.select("pickup_zip", "dropoff_zip").where(df_ej01.pickup_zip != df_ej01.dropoff_zip).groupBy("pickup_zip", "dropoff_zip").agg(count("*").alias("num_viajes_ruta")).orderBy("num_viajes_ruta", ascending=False)

display(df_ej07.limit(5))


In [0]:
# Agrupamos por destino y mostramos los que tienen mas de 100 viajes y también muestra su tarifa promedio
df_destinos_rentables = df_ej01.groupBy("dropoff_zip").agg(
    round(avg("fare_amount"),2).alias("tarifa_promedio"),
    count("dropoff_zip").alias("viajes")
)

display(df_destinos_rentables[df_destinos_rentables["viajes"] > 100])